In [1]:
# ============================================================
# SUPERSTORE DATA GENERATOR v2.0 — INDUSTRY LEVEL
# Description: Generates realistic, business-driven retail data
#              with seasonal trends, regional preferences,
#              customer behavior, and smarter pricing logic. 
# ============================================================

# ─────────────────────────────────────────────────────────────
# STEP 1: IMPORT LIBRARIES
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import random
from faker import Faker

fake = Faker()
random.seed(42)   # for reproducibility

# ─────────────────────────────────────────────────────────────
# STEP 2: BASE LISTS
# ─────────────────────────────────────────────────────────────
categories = {
    "Furniture":       ["Office Chair", "Study Table", "Sofa", "Bookshelf", "Dining Table"],
    "Office Supplies": ["Pen", "Notebook", "Stapler", "File Folder", "Calculator"],
    "Electronics":     ["Laptop", "Keyboard", "Mouse", "Headphones", "Monitor"],
    "Grocery":         ["Rice Bag", "Cooking Oil", "Sugar", "Snacks", "Juice Pack"]
}

regions          = ["North", "South", "East", "West"]
payment_modes    = ["Cash", "Credit Card", "UPI", "Net Banking"]
customer_segments = ["Consumer", "Corporate", "Home Office"]
order_priorities = ["High", "Medium", "Low"]

# ─────────────────────────────────────────────────────────────
# STEP 3: REPEAT CUSTOMER POOL
# (30% of orders come from a small group → realistic repeat buyers)
# ─────────────────────────────────────────────────────────────
repeat_customer_ids = [f"CUST{i}" for i in range(100, 131)]  # 30 loyal customers

# ─────────────────────────────────────────────────────────────
# STEP 4: GENERATE RECORDS
# ─────────────────────────────────────────────────────────────
records = []

for i in range(1000):
    order_id   = f"ORD{1000 + i}"
    order_date = fake.date_between(start_date='-2y', end_date='today')
    ship_date  = order_date + pd.Timedelta(days=random.randint(1, 7))

    # ── Customer (with repeat-buyer logic) ──────────────────
    if random.random() < 0.30:
        customer_id = random.choice(repeat_customer_ids)   # loyal customer
    else:
        customer_id = f"CUST{random.randint(200, 999)}"    # one-time buyer

    customer_name    = fake.name()
    customer_segment = random.choice(customer_segments)

    # ── Product ─────────────────────────────────────────────
    category     = random.choice(list(categories.keys()))
    product_name = random.choice(categories[category])
    product_id   = f"PROD{random.randint(1000, 9999)}"

    # ── Location ────────────────────────────────────────────
    region = random.choice(regions)
    state  = fake.state()
    city   = fake.city()

    # ─────────────────────────────────────────────────────────
    # IMPROVEMENT 1: CATEGORY-BASED PRICING LOGIC
    # Different categories have very different price ranges in
    # real retail — electronics are expensive, groceries cheap.
    # ─────────────────────────────────────────────────────────
    if category == "Electronics":
        unit_price = random.randint(2000, 50000)
    elif category == "Furniture":
        unit_price = random.randint(3000, 20000)
    elif category == "Office Supplies":
        unit_price = random.randint(50, 500)
    else:  # Grocery
        unit_price = random.randint(20, 1000)

    # ─────────────────────────────────────────────────────────
    # IMPROVEMENT 2: BASE QUANTITY
    # ─────────────────────────────────────────────────────────
    quantity = random.randint(1, 10)

    # ─────────────────────────────────────────────────────────
    # IMPROVEMENT 3: SEASONAL TRENDS
    # Festival months (Oct, Nov, Dec) → higher buying volume
    # Weekends → people shop more
    # ─────────────────────────────────────────────────────────
    if order_date.month in [10, 11, 12]:          # festival season boost
        quantity += random.randint(2, 5)
    if order_date.weekday() >= 5:                  # Saturday/Sunday boost
        quantity += random.randint(1, 3)

    # ─────────────────────────────────────────────────────────
    # IMPROVEMENT 4: REGION-BASED PREFERENCES
    # North India → tech-savvy → more Electronics
    # South India → food-focused → more Grocery
    # ─────────────────────────────────────────────────────────
    if region == "North" and category == "Electronics":
        quantity += 2
    if region == "South" and category == "Grocery":
        quantity += 3

    # ─────────────────────────────────────────────────────────
    # IMPROVEMENT 5: REALISTIC RETURN LOGIC
    # Electronics break/disappoint more often → 15% return rate
    # Other categories → 5% return rate
    # Pending/Cancelled added for real-world flavour
    # ─────────────────────────────────────────────────────────
    if category == "Electronics":
        delivery = random.choices(
            ["Delivered", "Returned", "Pending", "Cancelled"],
            weights=[75, 15, 7, 3]
        )[0]
    else:
        delivery = random.choices(
            ["Delivered", "Returned", "Pending", "Cancelled"],
            weights=[85, 5, 7, 3]
        )[0]

    # ── Financials ──────────────────────────────────────────
    discount     = random.choice([0, 5, 10, 15, 20])
    sales_amount = quantity * unit_price * (1 - discount / 100)
    cost_price   = sales_amount * random.uniform(0.6, 0.9)
    profit       = sales_amount - cost_price

    # ─────────────────────────────────────────────────────────
    # IMPROVEMENT 6: NEW BUSINESS COLUMNS
    # Profit Margin % → profitability health metric
    # Shipping Cost   → logistics cost (affects net profit)
    # Order Priority  → operational urgency
    # ─────────────────────────────────────────────────────────
    profit_margin  = round((profit / sales_amount) * 100, 2) if sales_amount > 0 else 0
    shipping_cost  = random.randint(50, 500)
    order_priority = random.choice(order_priorities)

    # ── Inventory ───────────────────────────────────────────
    stock_left = random.randint(0, 50)
    if stock_left < 10:
        auto_reorder    = "Yes"
        reorder_quantity = random.randint(20, 50)
    else:
        auto_reorder    = "No"
        reorder_quantity = 0

    # ── Supplier ────────────────────────────────────────────
    supplier_name  = fake.company()
    supplier_email = fake.company_email()
    payment_mode   = random.choice(payment_modes)

    # ── Append Record ───────────────────────────────────────
    records.append({
        "Order ID":            order_id,
        "Order Date":          order_date,
        "Ship Date":           ship_date,
        "Customer ID":         customer_id,
        "Customer Name":       customer_name,
        "Customer Segment":    customer_segment,
        "Product ID":          product_id,
        "Product Name":        product_name,
        "Category":            category,
        "Region":              region,
        "State":               state,
        "City":                city,
        "Quantity":            quantity,
        "Unit Price":          unit_price,
        "Discount (%)":        discount,
        "Sales Amount":        round(sales_amount, 2),
        "Cost Price":          round(cost_price, 2),
        "Profit":              round(profit, 2),
        "Profit Margin (%)":   profit_margin,        # NEW
        "Shipping Cost":       shipping_cost,         # NEW
        "Order Priority":      order_priority,        # NEW
        "Payment Mode":        payment_mode,
        "Delivery Status":     delivery,
        "Supplier Name":       supplier_name,
        "Supplier Email":      supplier_email,
        "Stock Left":          stock_left,
        "Auto Reorder":        auto_reorder,
        "Reorder Quantity":    reorder_quantity,
    })

# ─────────────────────────────────────────────────────────────
# STEP 5: SAVE TO CSV
# ─────────────────────────────────────────────────────────────
df = pd.DataFrame(records)

try:
    df.to_csv("Superstore_Management_System_v2.csv", index=False)
    print("✅ Dataset generated successfully!")
    print(f"   Shape : {df.shape}")
    print(f"   Columns: {list(df.columns)}")
except PermissionError:
    print("❌ Please close the file if it's open in Excel or Power BI.")

✅ Dataset generated successfully!
   Shape : (1000, 28)
   Columns: ['Order ID', 'Order Date', 'Ship Date', 'Customer ID', 'Customer Name', 'Customer Segment', 'Product ID', 'Product Name', 'Category', 'Region', 'State', 'City', 'Quantity', 'Unit Price', 'Discount (%)', 'Sales Amount', 'Cost Price', 'Profit', 'Profit Margin (%)', 'Shipping Cost', 'Order Priority', 'Payment Mode', 'Delivery Status', 'Supplier Name', 'Supplier Email', 'Stock Left', 'Auto Reorder', 'Reorder Quantity']
